# CrewAI Competitor Intelligence Crew

**Project:** Multi-Agent Competitive Intelligence & Weekly Memo Generator  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent competitive intelligence crew** that monitors competitors across three dimensions — features, pricing, and launches — then synthesizes everything into a single weekly executive memo.

### The Agent Roster

| # | Agent | Specialization | Deliverable |
|---|-------|---------------|------------|
| 1 | **Feature Tracker** | Product feature monitoring | Feature comparison matrix with gap analysis |
| 2 | **Pricing Analyst** | Pricing & packaging intelligence | Pricing benchmark report with positioning map |
| 3 | **Launch Monitor** | New product/release detection | Launch activity timeline with impact assessment |
| 4 | **Memo Writer** | Executive synthesis | Weekly competitive intelligence memo |

### Agent Specialization — The Core Concept

```
Competitive Landscape (raw signals)
         │
    ┌────┼────────────┐
    ▼    ▼            ▼
┌───────┐ ┌────────┐ ┌──────────┐
│Feature│ │Pricing │ │ Launch   │   Three SPECIALISTS
│Tracker│ │Analyst │ │ Monitor  │   each with a narrow lens
└───┬───┘ └───┬────┘ └────┬─────┘
    │         │           │
    └─────────┼───────────┘
              ▼
     ┌────────────────┐
     │  Memo Writer   │   One GENERALIST
     │  (synthesizer) │   who sees the full picture
     └────────────────┘
              │
              ▼
     Weekly Intel Memo
```

### Why Agent Specialization Matters

A single "competitive analyst" agent asked to track features AND pricing AND launches produces shallow, unfocused output because:

1. **Attention dilution** — broad prompts cause the LLM to spread attention thinly across all dimensions
2. **Missing vocabulary** — feature analysis needs technical language; pricing analysis needs financial language; launch monitoring needs timeline language
3. **Conflicting depth** — you can't go deep on pricing models while also cataloging 20 new features
4. **No cross-checking** — a single agent can't validate its own blind spots

**Specialization solves this** by giving each agent:
- A **narrow scope** (one dimension of competitive intelligence)
- A **domain-specific vocabulary** (embedded in the backstory)
- A **structured output format** (tailored to the dimension)
- An **explicit analytical framework** (so it knows HOW to analyze, not just WHAT)

The Memo Writer then acts as a **generalist synthesizer** — the only agent who sees all three specialist reports and fuses them into one coherent narrative.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All agents share the same local model. Specialization comes from role design, not model differences.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### The Anatomy of Agent Specialization

Specialization is achieved through four design levers — each one narrows what the agent focuses on and how it thinks:

| Lever | What It Controls | Example |
|-------|-----------------|---------|
| **Role** | Identity & scope | "Feature Tracker" vs. "Competitive Analyst" |
| **Goal** | Success criteria | "Identify 15+ feature differences" vs. "Analyze competitors" |
| **Backstory** | Domain vocabulary & methodology | Technical product comparison frameworks |
| **Task description** | Output structure & depth | Feature matrix with gap ratings |

**Anti-pattern:** Generic roles like "Competitive Intelligence Analyst" that try to cover everything. The LLM has no way to prioritize and produces average output across all dimensions.

**Best practice:** Each specialist agent should be able to answer: *"What ONE thing am I the expert on, and what is the structured format I deliver it in?"*

### 3.1 Feature Tracker Agent

**Specialization:** Product feature monitoring and gap analysis.

**What makes this agent specialized:**
- **Narrow scope:** Only tracks product features and capabilities — ignores pricing, marketing, partnerships
- **Domain vocabulary:** Uses product management terminology (feature parity, differentiation, table stakes, competitive moats)
- **Structured output:** Feature comparison matrix with gap ratings, not prose paragraphs
- **Analytical framework:** Categorizes features by tier (table stakes, differentiators, innovations) and rates competitive gaps

In [ ]:
feature_tracker = Agent(
    role="Competitive Feature Tracking Specialist",
    goal=(
        "Track and compare product features across competitors. Build a detailed "
        "feature comparison matrix, identify feature gaps (where competitors lead), "
        "feature advantages (where we lead), and emerging capabilities that could "
        "shift the competitive landscape. Categorize every feature as table stakes, "
        "differentiator, or innovation."
    ),
    backstory=(
        "You are a product intelligence analyst with 10 years of experience tracking "
        "competitive feature sets for B2B SaaS companies. You maintain living feature "
        "comparison matrices that product teams use for roadmap planning. Your methodology: "
        "(1) Feature Inventory — catalog every feature across all competitors in a standardized "
        "taxonomy, (2) Gap Analysis — for each feature, rate our position: LEADING (we have it, "
        "they don't), PARITY (comparable), LAGGING (they have it, we don't), ABSENT (neither "
        "has it but market needs it), (3) Tier Classification — Table Stakes (must-have to "
        "compete), Differentiator (creates preference), Innovation (new category or approach), "
        "(4) Velocity Tracking — how fast is each competitor shipping? Release cadence, beta "
        "features, public roadmap commitments. You focus exclusively on product capabilities — "
        "you do NOT analyze pricing, marketing, or business model. Your output is always a "
        "structured feature matrix, never a narrative essay."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {feature_tracker.role}")

### 3.2 Pricing Analyst Agent

**Specialization:** Pricing models, packaging, and monetization strategy analysis.

**What makes this agent specialized:**
- **Narrow scope:** Only analyzes pricing, packaging, and monetization — ignores feature details and launch timing
- **Domain vocabulary:** Uses pricing strategy terminology (ACV, ARR, price-to-value ratio, pricing power, GTM friction)
- **Structured output:** Pricing comparison table with positioning map, not feature lists
- **Analytical framework:** Evaluates pricing models (per-seat, usage-based, freemium), packaging tiers, and discount patterns

In [ ]:
pricing_analyst = Agent(
    role="Competitive Pricing & Monetization Analyst",
    goal=(
        "Analyze competitor pricing models, packaging structures, and monetization "
        "strategies. Map the pricing landscape on a positioning grid, identify pricing "
        "power indicators, compare price-to-value ratios, and flag any recent pricing "
        "changes or packaging experiments by competitors."
    ),
    backstory=(
        "You are a pricing strategy specialist with 8 years of experience at a SaaS "
        "pricing consultancy (Simon-Kucher, Bain pricing practice). You have redesigned "
        "pricing for 50+ B2B products and your competitive pricing analyses have directly "
        "influenced $200M+ in ARR decisions. Your framework: (1) Model Comparison — for each "
        "competitor, document the pricing model (per-seat, per-usage, flat-rate, freemium, "
        "open-core), tiers offered, and entry price, (2) Packaging Analysis — what features "
        "are gated behind each tier? Where are the upgrade triggers? (3) Price-to-Value Map — "
        "plot competitors on a 2x2 grid of price vs. perceived value to identify positioning "
        "(premium, value, penetration, skimming), (4) Pricing Signals — recent price increases, "
        "new tiers, free-to-paid conversions, enterprise pricing shifts, discount patterns at "
        "quarter-end, (5) Monetization Health — estimate competitor ACV/ARR where possible, "
        "identify whether pricing creates GTM friction (too complex, too expensive for POC). "
        "You focus exclusively on pricing and monetization. Feature details only matter to you "
        "insofar as they explain tier boundaries."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {pricing_analyst.role}")

### 3.3 Launch Monitor Agent

**Specialization:** Detecting and assessing new product launches, releases, and announcements.

**What makes this agent specialized:**
- **Narrow scope:** Only tracks launches, releases, and public announcements — not ongoing feature sets or pricing
- **Domain vocabulary:** Uses launch/release terminology (GA, beta, preview, EOL, breaking change, migration path)
- **Structured output:** Timeline of launch events with impact ratings, not feature comparisons
- **Analytical framework:** Classifies launches by type (new product, major version, feature release, acquisition) and assesses competitive impact

In [ ]:
launch_monitor = Agent(
    role="Product Launch & Release Intelligence Monitor",
    goal=(
        "Detect and assess all competitor product launches, major releases, "
        "acquisitions, partnerships, and strategic announcements from the past "
        "week. For each event, classify its type, assess competitive impact on "
        "our product, and identify required responses."
    ),
    backstory=(
        "You are a competitive intelligence analyst specializing in product launch "
        "monitoring. You have tracked launch activity for 7 years across the SaaS "
        "landscape, filing weekly launch intelligence reports that product, sales, and "
        "executive teams rely on. Your monitoring framework: (1) Event Detection — scan "
        "for product launches, GA releases, beta announcements, feature deprecations, "
        "SDK/API changes, acquisitions, strategic partnerships, and executive hires, "
        "(2) Event Classification — New Product (net-new offering), Major Version (v2, "
        "platform shift), Feature Release (incremental capability), Ecosystem Play "
        "(integration, marketplace, API), Strategic Move (acquisition, partnership, "
        "fundraise), (3) Impact Assessment — for each event, rate the competitive impact: "
        "Critical (directly threatens our core value prop), Significant (affects our "
        "positioning), Moderate (worth tracking), Low (tangential). (4) Response "
        "Recommendation — what should we do? Ignore, Monitor, Respond (messaging), Counter "
        "(build/ship), or Escalate (strategic review). You focus on EVENTS and TIMING, not "
        "ongoing feature parity or pricing. Your output is a chronological launch activity "
        "timeline with impact ratings."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {launch_monitor.role}")

### 3.4 Memo Writer Agent

**Specialization:** Executive synthesis and communication — the generalist who turns three specialist reports into one coherent memo.

**What makes this agent different from the specialists:**
- **Broad scope:** Sees ALL three specialist reports and connects insights across dimensions
- **Audience-aware:** Writes for busy executives, not product managers or analysts
- **Action-oriented:** Every section ends with a recommendation, not just an observation
- **Format-specific:** Produces a memo with a fixed structure (TL;DR, sections, action items) that executives expect

**Why the synthesizer role requires different skills:** The specialists go deep on one dimension. The memo writer goes wide — connecting a pricing change to a feature gap to a launch timeline to tell a coherent competitive story.

In [ ]:
memo_writer = Agent(
    role="Competitive Intelligence Memo Writer & Executive Synthesizer",
    goal=(
        "Synthesize the three specialist reports (features, pricing, launches) into "
        "a single weekly competitive intelligence memo for the executive team. The memo "
        "must be concise (readable in 5 minutes), action-oriented (every insight has a "
        "recommendation), and prioritized (most important developments first)."
    ),
    backstory=(
        "You are a chief of staff and competitive intelligence lead at a growth-stage "
        "SaaS company. You write the weekly competitive intel memo that the CEO, VP Product, "
        "VP Sales, and VP Marketing all read Monday morning. You've written 200+ weekly memos "
        "and have refined the format based on executive feedback. Your memo structure: "
        "(1) TL;DR — 3-5 bullet executive summary with the week's most important competitive "
        "developments, (2) Critical Alerts — must-respond items that need executive attention "
        "this week, (3) Feature Landscape — synthesize the feature tracker's matrix into 'so "
        "what' insights for product roadmap, (4) Pricing Intelligence — translate the pricing "
        "analyst's data into positioning implications for sales and marketing, (5) Launch "
        "Activity — highlight launches that affect our narrative and pipeline, (6) Action Items — "
        "specific, assigned, time-bound actions required (format: [Owner] Action by [Date]), "
        "(7) Watch List — items to monitor but not act on yet. Your writing style: no jargon, "
        "no hedging, every sentence either informs a decision or recommends an action. You "
        "connect dots between specialist reports — e.g., linking a competitor's pricing change "
        "to their new feature launch to predict their strategic direction."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {memo_writer.role}")

### Agent Roster — Specialization Map

In [ ]:
agents = [feature_tracker, pricing_analyst, launch_monitor, memo_writer]
specializations = [
    ("Features & Capabilities", "Feature matrix + gap analysis"),
    ("Pricing & Monetization", "Pricing map + packaging benchmark"),
    ("Launches & Announcements", "Launch timeline + impact ratings"),
    ("Executive Synthesis", "Weekly memo + action items"),
]

print("=" * 70)
print("COMPETITOR INTELLIGENCE CREW — SPECIALIZATION MAP")
print("=" * 70)
for i, (agent, (spec, output)) in enumerate(zip(agents, specializations), 1):
    role_type = "SPECIALIST" if i <= 3 else "GENERALIST"
    print(f"\n[{i}] {agent.role}")
    print(f"    Type: {role_type}")
    print(f"    Specialization: {spec}")
    print(f"    Deliverable: {output}")
print("\n" + "=" * 70)
print("\nPattern: 3 specialists (deep, narrow) → 1 generalist (wide, synthesized)")

## 4. Define Tasks & Intelligence Pipeline

### Specialization in Task Design

Agent specialization is reinforced through task design. Each specialist task:
1. Defines the **exact scope boundary** — what to analyze and what to explicitly ignore
2. Requires a **structured output format** — matrix, table, timeline (not free-form prose)
3. Provides **competitive context** — the specific competitors and our product to compare against
4. Sets **depth expectations** — number of features to track, pricing dimensions to compare

The memo writer's task is structurally different — it has a **fixed template** and its job is to **connect, prioritize, and recommend**, not to generate new analysis.

### 4.1 Configure the Competitive Landscape

In [ ]:
# =====================================================
# CONFIGURE YOUR COMPETITIVE LANDSCAPE HERE
# =====================================================
OUR_PRODUCT = {
    "name": "DataForge",
    "category": "Data Integration & ETL Platform",
    "description": (
        "Cloud-native data pipeline platform for building, orchestrating, and "
        "monitoring ETL/ELT workflows. Supports 150+ connectors, SQL and Python "
        "transformations, and real-time streaming."
    ),
    "pricing": "$99/mo (Starter, 5 pipelines), $499/mo (Pro, unlimited), Enterprise custom",
    "recent_updates": "Shipped reverse ETL, dbt integration, and SOC2 compliance last month",
}

COMPETITORS = [
    {
        "name": "Fivetran",
        "focus": "Managed ELT with 500+ connectors",
        "pricing": "Usage-based (MAR), starts ~$1/credit",
    },
    {
        "name": "Airbyte",
        "focus": "Open-source ELT with cloud and self-hosted options",
        "pricing": "Free (open-source), Cloud $2.50/credit",
    },
    {
        "name": "Stitch (by Talend)",
        "focus": "Simple cloud ETL for SMBs",
        "pricing": "Free tier, Standard ($100/mo), Advanced ($1,000+/mo)",
    },
    {
        "name": "dbt Cloud",
        "focus": "Transformation layer (T in ELT) with analytics engineering",
        "pricing": "Free (developer), Team ($100/seat/mo), Enterprise custom",
    },
]

WEEK_CONTEXT = "Week of April 7-11, 2025"
# =====================================================

print("OUR PRODUCT")
print("=" * 50)
for k, v in OUR_PRODUCT.items():
    print(f"  {k.replace('_', ' ').title():<20} {v}")

print(f"\nCOMPETITORS ({len(COMPETITORS)})")
print("=" * 50)
for c in COMPETITORS:
    print(f"  {c['name']:<20} {c['focus']}")
    print(f"  {'':20} Pricing: {c['pricing']}")
print(f"\nReporting period: {WEEK_CONTEXT}")

In [ ]:
# Build the competitive context string used by all tasks
landscape = (
    f"=== OUR PRODUCT ===\n"
    f"Name: {OUR_PRODUCT['name']}\n"
    f"Category: {OUR_PRODUCT['category']}\n"
    f"Description: {OUR_PRODUCT['description']}\n"
    f"Pricing: {OUR_PRODUCT['pricing']}\n"
    f"Recent Updates: {OUR_PRODUCT['recent_updates']}\n"
    f"\n=== COMPETITORS ===\n"
)
for c in COMPETITORS:
    landscape += f"- {c['name']}: {c['focus']} | Pricing: {c['pricing']}\n"

landscape += f"\nReporting Period: {WEEK_CONTEXT}"
print(landscape)

### 4.2 Task 1 — Feature Tracking (Specialist)

**Scope boundary:** Product features and capabilities ONLY. No pricing, no launch timing, no marketing.

**Output format:** Feature comparison matrix with gap ratings per category.

In [ ]:
task_features = Task(
    description=(
        f"Conduct a feature-by-feature competitive analysis for this landscape:\n\n"
        f"{landscape}\n\n"
        "YOUR SCOPE: Product features and capabilities ONLY.\n"
        "NOT YOUR SCOPE: Pricing, launches, partnerships, marketing.\n\n"
        "Deliverables:\n"
        "1. **Feature Comparison Matrix**: Compare features across these categories:\n"
        "   - Data Connectors (count, types, custom connector support)\n"
        "   - Transformation Capabilities (SQL, Python, dbt, visual builder)\n"
        "   - Orchestration (scheduling, dependencies, error handling, retries)\n"
        "   - Real-Time / Streaming (CDC, event-driven, latency)\n"
        "   - Monitoring & Observability (data quality, lineage, alerting)\n"
        "   - Security & Compliance (SOC2, HIPAA, encryption, RBAC)\n"
        "   - Reverse ETL / Activation (sync to SaaS tools)\n\n"
        "2. **Gap Rating** per category: LEADING / PARITY / LAGGING / ABSENT\n"
        "   - Rate {OUR_PRODUCT['name']} vs. each competitor\n\n"
        "3. **Feature Tier Classification** for each feature:\n"
        "   - Table Stakes (must-have to compete in this market)\n"
        "   - Differentiator (creates buyer preference)\n"
        "   - Innovation (new approach not yet standard)\n\n"
        "4. **Top 3 Feature Gaps**: Where competitors are ahead and it matters most\n"
        "5. **Top 3 Feature Advantages**: Where we lead and should amplify\n"
        "6. **Emerging Capabilities**: Features in beta or on public roadmaps\n"
    ),
    expected_output=(
        "A feature comparison matrix across 7 categories with gap ratings "
        "(LEADING/PARITY/LAGGING/ABSENT), tier classification, top gaps, "
        "top advantages, and emerging capabilities."
    ),
    agent=feature_tracker,
)

print(f"Task created: Feature Tracking -> {task_features.agent.role}")

### 4.3 Task 2 — Pricing Intelligence (Specialist)

**Scope boundary:** Pricing models, packaging, and monetization ONLY. Features only matter for tier gating.

**Output format:** Pricing comparison table with positioning map.

In [ ]:
task_pricing = Task(
    description=(
        f"Analyze the pricing and monetization landscape for:\n\n"
        f"{landscape}\n\n"
        "YOUR SCOPE: Pricing models, packaging, tiers, and monetization strategy ONLY.\n"
        "NOT YOUR SCOPE: Feature details (beyond tier gating), launch timing, marketing.\n\n"
        "Deliverables:\n"
        "1. **Pricing Model Comparison Table**:\n"
        "   For each competitor, document:\n"
        "   - Pricing model type (per-seat, per-usage/credit, flat-rate, open-core)\n"
        "   - Entry price (lowest paid tier)\n"
        "   - Mid-market price (typical 10-50 user deployment)\n"
        "   - Enterprise model (custom, negotiated, published)\n"
        "   - Free tier / trial availability\n\n"
        "2. **Packaging Analysis**:\n"
        "   - What features gate the upgrade from free → paid → enterprise?\n"
        "   - Where are the monetization friction points?\n"
        "   - Which competitor has the smoothest land-and-expand motion?\n\n"
        "3. **Price-to-Value Positioning Map**:\n"
        "   - Plot competitors on a Price vs. Perceived Value 2x2\n"
        "   - Identify: Premium players, Value players, Overpriced, Underpriced\n\n"
        "4. **Pricing Signals This Week**:\n"
        "   - Any recent pricing changes, new tiers, or packaging experiments?\n"
        "   - Quarter-end discount patterns?\n\n"
        "5. **Pricing Recommendation for {OUR_PRODUCT['name']}**:\n"
        "   - Is our pricing positioned correctly vs. competitors?\n"
        "   - Should we adjust entry price, add a tier, or change the model?\n"
    ),
    expected_output=(
        "A pricing intelligence report with model comparison table, packaging "
        "analysis, price-to-value positioning map, recent pricing signals, and "
        "pricing recommendations."
    ),
    agent=pricing_analyst,
)

print(f"Task created: Pricing Intelligence -> {task_pricing.agent.role}")

### 4.4 Task 3 — Launch Activity (Specialist)

**Scope boundary:** Launches, releases, announcements, and strategic moves ONLY.

**Output format:** Chronological launch timeline with impact/response ratings.

In [ ]:
task_launches = Task(
    description=(
        f"Monitor competitor launch and release activity for:\n\n"
        f"{landscape}\n\n"
        "YOUR SCOPE: Product launches, releases, announcements, and strategic moves ONLY.\n"
        "NOT YOUR SCOPE: Ongoing feature parity, pricing analysis, marketing campaigns.\n\n"
        "Scan for these event types (for the reporting period):\n"
        "- New product launches or GA releases\n"
        "- Major version updates or platform changes\n"
        "- Beta/preview announcements\n"
        "- Feature deprecations or EOL announcements\n"
        "- Acquisitions, mergers, or strategic partnerships\n"
        "- Significant funding rounds\n"
        "- Key executive hires or departures\n"
        "- Conference announcements or keynote reveals\n\n"
        "For EACH event detected, provide:\n"
        "1. **Date** and **Competitor**\n"
        "2. **Event Type**: New Product / Major Version / Feature Release / Ecosystem Play / "
        "   Strategic Move / Leadership Change\n"
        "3. **Description**: What happened in 2-3 sentences\n"
        "4. **Impact Rating**: Critical / Significant / Moderate / Low\n"
        "5. **Response Recommendation**: Ignore / Monitor / Respond (messaging) / Counter "
        "   (build) / Escalate (strategy review)\n"
        "6. **Affected Teams**: Product / Sales / Marketing / Executive\n\n"
        "Organize as a chronological timeline with most impactful events first.\n"
        "End with a **Weekly Launch Velocity Score**: how active was the competitive "
        "landscape this week (Quiet / Normal / Active / Intense)?\n"
    ),
    expected_output=(
        "A launch activity timeline with events classified by type, rated by "
        "competitive impact, with response recommendations and a weekly "
        "velocity score."
    ),
    agent=launch_monitor,
)

print(f"Task created: Launch Activity -> {task_launches.agent.role}")

### 4.5 Task 4 — Weekly Memo (Generalist Synthesizer)

**Scope:** EVERYTHING — the memo writer sees all three specialist reports and synthesizes them into a single executive memo.

**Key job:** Connect the dots. A feature release (from launch monitor) may explain a pricing change (from pricing analyst) which creates a feature gap (from feature tracker). Only the memo writer sees all three and can tell the full story.

In [ ]:
task_memo = Task(
    description=(
        f"Write the weekly competitive intelligence memo for {OUR_PRODUCT['name']}.\n\n"
        f"Reporting Period: {WEEK_CONTEXT}\n\n"
        "You have received three specialist reports:\n"
        "1. Feature Tracker — feature comparison matrix with gaps and advantages\n"
        "2. Pricing Analyst — pricing landscape with positioning map\n"
        "3. Launch Monitor — launch activity timeline with impact ratings\n\n"
        "Write the weekly memo using this EXACT structure:\n\n"
        "## TL;DR\n"
        "3-5 bullet executive summary. Most important development first.\n\n"
        "## Critical Alerts\n"
        "Items that need executive attention THIS WEEK. If none, say 'No critical "
        "alerts this week.'\n\n"
        "## Feature Landscape\n"
        "Synthesize the feature matrix into strategic implications. Focus on:\n"
        "- Feature gaps that affect win rates\n"
        "- Advantages we should amplify in sales messaging\n"
        "- Emerging capabilities that need roadmap discussion\n\n"
        "## Pricing Intelligence\n"
        "Translate pricing data into business impact:\n"
        "- How does our pricing compare in active deals?\n"
        "- Are competitors using pricing as a weapon?\n"
        "- Should we adjust our pricing or packaging?\n\n"
        "## Launch Activity\n"
        "Highlight launches that affect our narrative and pipeline.\n"
        "Connect launches to feature gaps and pricing shifts.\n\n"
        "## Action Items\n"
        "Specific, assigned, time-bound actions:\n"
        "Format: [OWNER] Action description — by [DATE]\n"
        "Include 4-6 action items for Product, Sales, Marketing, and Executive.\n\n"
        "## Watch List\n"
        "Items to monitor but not act on yet. Include trigger conditions.\n\n"
        "WRITING RULES:\n"
        "- No jargon. Write for busy executives.\n"
        "- Every sentence informs a decision or recommends an action.\n"
        "- Connect dots across the three specialist reports.\n"
        "- Use bold for key findings and competitor names.\n"
        "- Total memo: 800-1200 words max.\n"
    ),
    expected_output=(
        "A weekly competitive intelligence memo with TL;DR, critical alerts, "
        "feature landscape, pricing intelligence, launch activity, action items, "
        "and watch list. 800-1200 words, action-oriented, executive-readable."
    ),
    agent=memo_writer,
)

print(f"Task created: Weekly Memo -> {task_memo.agent.role}")

### Intelligence Pipeline Visualization

In [ ]:
tasks = [task_features, task_pricing, task_launches, task_memo]
task_names = ["Feature Tracking", "Pricing Intelligence", "Launch Activity", "Weekly Memo"]
agent_types = ["SPECIALIST", "SPECIALIST", "SPECIALIST", "GENERALIST"]

print("COMPETITIVE INTELLIGENCE PIPELINE")
print("=" * 70)
for i, (name, task, atype) in enumerate(zip(task_names, tasks, agent_types)):
    print(f"  [{atype}] {name}")
    print(f"    Agent: {task.agent.role}")
    if i < 3:
        scope = ["Features & capabilities", "Pricing & monetization", "Launches & events"][i]
        print(f"    Scope: {scope} ONLY")
    else:
        print(f"    Scope: ALL specialist reports → executive memo")
    if i < len(tasks) - 1:
        if i == 2:
            print(f"    {'':8}│")
            print(f"    {'':8}│  (all 3 specialist reports converge ↓)")
            print(f"    {'':8}│")
        else:
            print(f"    {'':8}│")
print("=" * 70)

## 5. Assemble and Run the Crew

### Orchestration Choice: Sequential with Convergent Synthesis

The three specialists run sequentially (they don't depend on each other, but sequential is simpler to implement). The memo writer runs last and sees all three reports.

**Why not parallel?** In CrewAI, `Process.sequential` is the default. The specialists COULD run in parallel since they're independent. We demonstrate parallel execution with explicit `context` in Section 10.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")
print(f"Specialists: {sum(1 for t in agent_types if t == 'SPECIALIST')}")
print(f"Generalists: {sum(1 for t in agent_types if t == 'GENERALIST')}")

In [ ]:
# Execute the competitive intelligence pipeline
print("=" * 70)
print(f"LAUNCHING COMPETITOR INTELLIGENCE CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Our Product: {OUR_PRODUCT['name']}")
print(f"Competitors: {', '.join(c['name'] for c in COMPETITORS)}")
print(f"Period: {WEEK_CONTEXT}")
print(f"Pipeline: Features -> Pricing -> Launches -> Memo")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Weekly Competitive Intelligence Memo

In [ ]:
print("=" * 70)
print(f"WEEKLY COMPETITIVE INTELLIGENCE MEMO — {WEEK_CONTEXT}")
print("=" * 70)
print(result.raw)

### 6.2 Specialist Reports (Raw)

In [ ]:
for atype, name, task in zip(agent_types, task_names, tasks):
    print("\n" + "=" * 70)
    print(f"[{atype}] {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Specialization Effectiveness Metrics

In [ ]:
print("SPECIALIZATION EFFECTIVENESS")
print("=" * 65)
print(f"{'Type':<12} {'Task':<22} {'Agent':<25} {'Output':>8}")
print("-" * 65)

total = 0
for atype, name, task in zip(agent_types, task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    agent_short = task.agent.role.split("&")[0].strip()[:23]
    print(f"{atype:<12} {name:<22} {agent_short:<25} {length:>6,}")

print("-" * 65)
print(f"{'':12} {'TOTAL':<47} {total:>6,}")

spec_total = sum(len(t.output.raw) for t, at in zip(tasks, agent_types)
                 if t.output and at == "SPECIALIST")
gen_total = sum(len(t.output.raw) for t, at in zip(tasks, agent_types)
                if t.output and at == "GENERALIST")
print(f"\nSpecialist output: {spec_total:,} chars (raw intelligence)")
print(f"Generalist output: {gen_total:,} chars (synthesized memo)")
if spec_total > 0:
    ratio = gen_total / spec_total
    print(f"Compression ratio: {ratio:.1%} (memo is {ratio:.0%} the size of raw reports)")

## 7. Save Intel Reports

In [ ]:
# Save the weekly memo
if task_memo.output:
    with open("weekly_memo.md", "w", encoding="utf-8") as f:
        f.write(f"# Competitive Intelligence Memo — {WEEK_CONTEXT}\n\n")
        f.write(task_memo.output.raw)
    print(f"Memo saved: weekly_memo.md ({len(task_memo.output.raw):,} chars)")

# Save all specialist reports
report_lines = [
    f"# Competitor Intelligence Report — {WEEK_CONTEXT}",
    f"**Product:** {OUR_PRODUCT['name']}",
    f"**Competitors:** {', '.join(c['name'] for c in COMPETITORS)}",
    f"**Generated:** {datetime.now().isoformat()}",
    "---",
]
for atype, name, task in zip(agent_types, task_names, tasks):
    report_lines.append(f"\n## [{atype}] {name}\n")
    report_lines.append(task.output.raw if task.output else "[No output]")
    report_lines.append("\n---")

report = "\n".join(report_lines)
with open("full_intel_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(f"Full report saved: full_intel_report.md ({len(report):,} chars)")

## 8. Experiment: Different Competitive Landscape

Re-run the same crew for a completely different product category to validate that specialization produces domain-appropriate output regardless of industry.

In [ ]:
# =====================================================
# SECOND LANDSCAPE — Different industry
# =====================================================
PRODUCT_2 = {
    "name": "DocuMind",
    "category": "AI Document Processing & Extraction",
    "description": (
        "AI-powered document understanding platform that extracts structured data "
        "from invoices, contracts, forms, and receipts using LLM + vision models."
    ),
    "pricing": "Pay-per-page: $0.10/page (Standard), $0.25/page (Premium with LLM extraction)",
    "recent_updates": "Added multi-language support (12 languages) and table extraction v2",
}

COMPETITORS_2 = [
    {"name": "AWS Textract", "focus": "Cloud OCR + form extraction", "pricing": "$1.50/1000 pages"},
    {"name": "Google Document AI", "focus": "Pre-trained document processors", "pricing": "$0.10-0.65/page by type"},
    {"name": "ABBYY Vantage", "focus": "Enterprise IDP with pre-built skills", "pricing": "Custom enterprise pricing"},
    {"name": "Rossum", "focus": "AI document gateway for transactional docs", "pricing": "Per-document, custom"},
]
WEEK_2 = "Week of April 7-11, 2025"

landscape2 = (
    f"=== OUR PRODUCT ===\n"
    f"Name: {PRODUCT_2['name']}\n"
    f"Category: {PRODUCT_2['category']}\n"
    f"Description: {PRODUCT_2['description']}\n"
    f"Pricing: {PRODUCT_2['pricing']}\n"
    f"Recent: {PRODUCT_2['recent_updates']}\n\n"
    f"=== COMPETITORS ===\n"
)
for c in COMPETITORS_2:
    landscape2 += f"- {c['name']}: {c['focus']} | {c['pricing']}\n"
landscape2 += f"\nPeriod: {WEEK_2}"

print(f"Product 2: {PRODUCT_2['name']} ({PRODUCT_2['category']})")
print(f"Competitors: {', '.join(c['name'] for c in COMPETITORS_2)}")

In [ ]:
task2_features = Task(
    description=(
        f"Feature analysis for:\n{landscape2}\n\n"
        "Scope: Features ONLY. Compare: OCR accuracy, document types supported, "
        "extraction capabilities (tables, handwriting, signatures), API design, "
        "batch processing, model customization, output formats. Rate gaps and "
        "classify feature tiers."
    ),
    expected_output="Feature comparison matrix with gap ratings for document AI category.",
    agent=feature_tracker,
)

task2_pricing = Task(
    description=(
        f"Pricing analysis for:\n{landscape2}\n\n"
        "Scope: Pricing ONLY. Compare per-page vs. per-document vs. volume-based "
        "models. Analyze free tier availability, volume discounts, enterprise "
        "pricing transparency. Build positioning map."
    ),
    expected_output="Pricing comparison with model analysis and positioning map.",
    agent=pricing_analyst,
)

task2_launches = Task(
    description=(
        f"Launch monitoring for:\n{landscape2}\n\n"
        "Scope: Launches/announcements ONLY. Track any new document processors, "
        "model updates, API changes, partnership announcements, or pricing changes "
        "this week. Rate impact and recommend responses."
    ),
    expected_output="Launch activity timeline with impact ratings.",
    agent=launch_monitor,
)

task2_memo = Task(
    description=(
        f"Write weekly competitive intelligence memo for {PRODUCT_2['name']}.\n"
        f"Period: {WEEK_2}\n\n"
        "Use the memo template: TL;DR, Critical Alerts, Feature Landscape, Pricing "
        "Intelligence, Launch Activity, Action Items, Watch List. 800-1200 words, "
        "executive-readable."
    ),
    expected_output="Weekly competitive intel memo for document AI landscape.",
    agent=memo_writer,
)

tasks_2 = [task2_features, task2_pricing, task2_launches, task2_memo]
print(f"Landscape 2 tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Product: {PRODUCT_2['name']}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"WEEKLY MEMO — {PRODUCT_2['name']}")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Landscapes

In [ ]:
print("COMPETITIVE INTELLIGENCE COMPARISON")
print("=" * 72)
header = f"{'Type':<12} {'Task':<22} {'DataForge':>14} {'DocuMind':>14}"
print(header)
print("-" * 72)

for atype, name, t1, t2 in zip(agent_types, task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{atype:<12} {name:<22} {len1:>11,} ch {len2:>11,} ch")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 72)
print(f"{'':12} {'TOTAL':<22} {total1:>11,} ch {total2:>11,} ch")

## 10. Advanced: Independent Specialists with Explicit Context

### The Ideal Specialization Pattern

The three specialists should work **independently** — each from the raw competitive landscape, not from each other's reports. This prevents the pricing analyst from being anchored by the feature tracker's framing.

```
                  ┌──> Feature Tracker  ──┐
Competitive       │                       │
Landscape  ───────┼──> Pricing Analyst  ──┼──> Memo Writer
                  │                       │
                  └──> Launch Monitor   ──┘
```

Using explicit `context`, we ensure:
- Each specialist sees ONLY the landscape (not other specialists' work)
- The memo writer sees ALL THREE specialist reports
- No specialist anchors another

In [ ]:
# Independent specialist tasks with explicit context

# A dummy "input" task is not needed — specialists receive the landscape
# in their task description. We use context to control what the memo writer sees.

ind_features = Task(
    description=(
        f"Feature analysis for:\n{landscape}\n\n"
        "Scope: Features ONLY. Build comparison matrix, rate gaps, classify tiers."
    ),
    expected_output="Feature comparison matrix with gap ratings.",
    agent=feature_tracker,
)

ind_pricing = Task(
    description=(
        f"Pricing analysis for:\n{landscape}\n\n"
        "Scope: Pricing ONLY. Compare models, build positioning map, flag signals."
    ),
    expected_output="Pricing comparison with positioning map.",
    agent=pricing_analyst,
    context=[],  # Empty context = no prior outputs, just the task description
)

ind_launches = Task(
    description=(
        f"Launch monitoring for:\n{landscape}\n\n"
        "Scope: Launches ONLY. Build timeline, rate impact, recommend responses."
    ),
    expected_output="Launch timeline with impact ratings.",
    agent=launch_monitor,
    context=[],  # Independent — no prior outputs
)

# Memo writer explicitly sees all three
ind_memo = Task(
    description=(
        f"Write weekly competitive intel memo for {OUR_PRODUCT['name']}.\n"
        f"Period: {WEEK_CONTEXT}\n\n"
        "Synthesize all three specialist reports. Use standard memo template: "
        "TL;DR, Critical Alerts, Feature Landscape, Pricing Intelligence, "
        "Launch Activity, Action Items, Watch List."
    ),
    expected_output="Weekly competitive intelligence memo.",
    agent=memo_writer,
    context=[ind_features, ind_pricing, ind_launches],  # Sees all three
)

ind_tasks = [ind_features, ind_pricing, ind_launches, ind_memo]

print("INDEPENDENT SPECIALIST FLOW")
print("=" * 60)
print("                 ┌──> Feature Tracker  ──┐")
print("  Landscape  ────┼──> Pricing Analyst  ──┼──> Memo Writer")
print("                 └──> Launch Monitor   ──┘")
print()
ind_names = ["Feature Tracking", "Pricing Intel", "Launch Activity", "Weekly Memo"]
for i, (name, task) in enumerate(zip(ind_names, ind_tasks), 1):
    ctx = task.context
    if ctx is None:
        ctx_str = "auto (all prior)"
    elif len(ctx) == 0:
        ctx_str = "NONE (independent)"
    else:
        ctx_str = " + ".join(t.agent.role.split("&")[0].strip()[:20] for t in ctx)
    print(f"  Task {i}: {name}")
    print(f"    Context: {ctx_str}")

In [ ]:
crew_ind = Crew(
    agents=agents,
    tasks=ind_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching independent-specialist crew — {datetime.now().strftime('%H:%M:%S')}")
result_ind = crew_ind.kickoff()
print(f"\nIndependent-specialist crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("INDEPENDENT SPECIALIST RESULTS")
print("=" * 65)
for name, task in zip(ind_names, ind_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = task.context
    if ctx is None:
        ctx_str = "auto"
    elif len(ctx) == 0:
        ctx_str = "independent"
    else:
        ctx_str = f"{len(ctx)} reports"
    print(f"  {name:<22} {length:>6,} chars | Context: {ctx_str}")

## 11. Key Takeaways

### What We Built
- A **4-agent competitive intelligence crew** with 3 specialists + 1 generalist synthesizer
- Ran it on **two competitive landscapes** (Data Integration + Document AI)
- Demonstrated **independent specialization** with explicit `context` to prevent anchoring

### Agent Specialization Concepts
1. **Narrow scope > broad scope** — Each specialist owns ONE dimension of competitive intelligence (features, pricing, or launches) and ignores everything else
2. **Structured outputs > prose** — Specialists deliver matrices, tables, and timelines; the generalist delivers a narrative memo
3. **Domain vocabulary matters** — The feature tracker uses product management language; the pricing analyst uses monetization language; the launch monitor uses release management language
4. **Specialists are independent; the generalist connects** — The memo writer is the ONLY agent who sees all three reports and connects insights across dimensions

### Specialization Design Patterns

| Pattern | Structure | When to Use |
|---------|-----------|-------------|
| **Specialist + Synthesizer** (this notebook) | N narrow agents → 1 wide agent | Monitoring tasks with multiple dimensions |
| **Sequential Specialists** | A → B → C (each builds on prior) | Analysis where each step depends on the previous |
| **Cross-Validating Specialists** | A sees B's output, B sees A's | When specialists might contradict each other |
| **Specialist with Critic** | Specialist → Dedicated Critic | When output quality is critical |

### Agent Design Lessons for Specialization
- **Feature Tracker:** Express the gap rating scale (LEADING/PARITY/LAGGING/ABSENT) in the backstory — it becomes the agent's analytical vocabulary
- **Pricing Analyst:** Restrict scope explicitly ("features only matter for tier gating") to prevent scope creep into feature analysis
- **Launch Monitor:** Define event types and impact ratings as enums — this forces structured, classifiable output
- **Memo Writer:** Specify word count and template sections — without constraints, synthesis agents either over-summarize or over-elaborate

### Production Tips
- Connect specialists to real data sources: GitHub release feeds (launch monitor), pricing page scrapers (pricing analyst), product changelog RSS (feature tracker)
- Use `output_pydantic` with typed schemas per specialist for structured JSON output
- Schedule weekly runs with a cron job and pipe the memo to Slack/email
- Add a **Customer Win/Loss Analyst** specialist that tracks competitive deal outcomes
- Enable `memory=True` to track competitor trends across weeks, building a longitudinal intelligence database